In [1]:
#This is program for count VLM research

In [3]:
import pandas as pd
import arxiv

In [5]:
client = arxiv.Client(
    page_size = 100,
    delay_seconds = 3.0,
    num_retries = 10,      # 間欠障害が前提なので回数を確保する
)
results = client.results(search)
def convert_to_str(vals):  # 文字列に変換
    if isinstance(vals, list):
        return list(map(str, vals))
    elif vals is None:
        return vals
    else:
        return str(vals)
        
def create_df(results):
    res0 = []
    for res in results:
        res_dict = dict(
            entry_id = res.entry_id,
            pdf_url = res.pdf_url,
            updated = res.updated, 
            published = res.published, 
            title = res.title, 
            authors = convert_to_str(res.authors),  # 文字列に変換
            summary = res.summary,
            comment = res.comment,
            journal_ref = res.journal_ref,
            doi = res.doi,
            primary_category = res.primary_category,
            categories = res.categories,
            links = convert_to_str(res.links),  # 文字列に変換
        )
        res0.append(res_dict)
    df = pd.DataFrame(res0)
    ndata = len(df)
    return df

df = create_df(results)


In [6]:

search = arxiv.Search(
    query = 'abs:"autonomous driving" AND (abs:"VLM" OR abs:"vision language model")',
    max_results = 1000,
    sort_by = arxiv.SortCriterion.SubmittedDate,
    sort_order = arxiv.SortOrder.Descending,
)
df = create_df(client.results(search))
print(len(df))
query = 'abs:"autonomous driving" AND (abs:"E2E" OR abs:"End to End" OR "End-to-End")',

382


In [11]:
import urllib.parse, urllib.request
import xml.etree.ElementTree as ET

NS = {'opensearch': 'http://a9.com/-/spec/opensearch/1.1/'}

def arxiv_total(query: str) -> int:
    url = ('http://export.arxiv.org/api/query?search_query='
           + urllib.parse.quote(query) + '&max_results=1')
    with urllib.request.urlopen(url) as r:
        root = ET.fromstring(r.read())
    return int(root.find('opensearch:totalResults', NS).text)

print(arxiv_total(query))

TypeError: quote_from_bytes() expected bytes

In [7]:
import numpy as np

def create_url(aid: str):
    apath = aid.split("/")[-1].split("v")[0]
    arxiv_id = f'ARXIV:{apath}'
    search_url = f'{semantic_base_url}/{arxiv_id}' + '?fields=' + fields 
    return arxiv_id

semantic_base_url = 'https://api.semanticscholar.org/graph/v1/paper'  # Semantic ScholarのベースURL
fields = 'citationCount,influentialCitationCount,referenceCount'  # サイトから取得する追加項
df['arxiv_id'] = df['entry_id'].apply(create_url)
tcol = fields.split(",") 
df[tcol] = np.nan  # 初期化

In [8]:
import requests
import time

HEADERS = {}   # APIキー取得後は {"x-api-key": API_KEY}

def post_batch(batch, max_retry=6):
    for attempt in range(max_retry):
        r = requests.post(semantic_base_url + '/batch',
                          params={'fields': fields}, json={"ids": batch},
                          headers=HEADERS, timeout=30)
        if r.status_code == 200:
            return r.json()
        if r.status_code in (429, 500, 502, 503, 504):
            wait = int(r.headers.get('Retry-After', 0)) or 5 * 2 ** attempt
            print(f'  {r.status_code}: retry in {wait}s (attempt {attempt+1})')
            time.sleep(wait)
            continue
        r.raise_for_status()
    return None


def get_citation_batch(df, start=0, batch_size=500):
    outs = []
    data = df['arxiv_id'].tolist()
    for i in range(start, len(data), batch_size):
        batch = data[i:i+batch_size]
        payload = post_batch(batch)
        if payload is None:
            print('Error: giving up. restart with start=', i)
            break
        rows = {pid: rec for pid, rec in zip(batch, payload) if rec is not None}
        if rows:
            outs.append(pd.DataFrame.from_dict(rows, orient="index"))
        time.sleep(5)
    if not outs:
        return df
    res = pd.concat(outs).rename_axis('arxiv_id').reset_index()
    return df.merge(res[['arxiv_id'] + tcol], on='arxiv_id', how='left')


start = 0  # 開始行
batch_size = 500  # バッチ数。500以下
df = get_citation_batch(df, start=start, batch_size=batch_size)

  429: retry in 5s (attempt 1)
  429: retry in 10s (attempt 2)
  429: retry in 20s (attempt 3)


In [9]:
df = df.drop(columns=[c + '_x' for c in tcol])
df = df.rename(columns={c + '_y': c for c in tcol})

In [13]:
pd.set_option("display.max_colwidth", None)

In [14]:
df.sort_values(['citationCount'],ascending=False).head(15)

,entry_id,pdf_url,updated,published,title,authors,summary,comment,journal_ref,doi,primary_category,categories,links,arxiv_id,citationCount,influentialCitationCount,referenceCount
371,http://arxiv.org/abs/2312.14150v3,https://arxiv.org/pdf/2312.14150v3,2025-01-16 10:57:44+00:00,2023-12-21 18:59:12+00:00,DriveLM: Driving with Graph Visual Question Answering,"[Chonghao Sima, Katrin Renz, Kashyap Chitta, Li Chen, Hanxue Zhang, Chengen Xie, Jens Beißwenger, Ping Luo, Andreas Geiger, Hongyang Li]","We study how vision-language models (VLMs) trained on web-scale data can be integrated into end-to-end driving systems to boost generalization and enable interactivity with human users. While recent approaches adapt VLMs to driving via single-round visual question answering (VQA), human drivers reason about decisions in multiple steps. Starting from the localization of key objects, humans estimate object interactions before taking actions. The key insight is that with our proposed task, Graph VQA, where we model graph-structured reasoning through perception, prediction and planning question-answer pairs, we obtain a suitable proxy task to mimic the human reasoning process. We instantiate datasets (DriveLM-Data) built upon nuScenes and CARLA, and propose a VLM-based baseline approach (DriveLM-Agent) for jointly performing Graph VQA and end-to-end driving. The experiments demonstrate that Graph VQA provides a simple, principled framework for reasoning about a driving scene, and DriveLM-Data provides a challenging benchmark for this task. Our DriveLM-Agent baseline performs end-to-end autonomous driving competitively in comparison to state-of-the-art driving-specific architectures. Notably, its benefits are pronounced when it is evaluated zero-shot on unseen objects or sensor configurations. We hope this work can be the starting point to shed new light on how to apply VLMs for autonomous driving. To facilitate future research, all code, data, and models are available to the public.",Accepted to ECCV 2024 as Oral paper,NaN,NaN,cs.CV,[cs.CV],"[https://arxiv.org/abs/2312.14150v3, https://arxiv.org/pdf/2312.14150v3]",ARXIV:2312.14150,609.0,86.0,101.0
369,http://arxiv.org/abs/2402.12289v5,https://arxiv.org/pdf/2402.12289v5,2024-06-25 17:55:35+00:00,2024-02-19 17:04:04+00:00,DriveVLM: The Convergence of Autonomous Driving and Large Vision-Language Models,"[Xiaoyu Tian, Junru Gu, Bailin Li, Yicheng Liu, Yang Wang, Zhiyong Zhao, Kun Zhan, Peng Jia, Xianpeng Lang, Hang Zhao]","A primary hurdle of autonomous driving in urban environments is understanding complex and long-tail scenarios, such as challenging road conditions and delicate human behaviors. We introduce DriveVLM, an autonomous driving system leveraging Vision-Language Models (VLMs) for enhanced scene understanding and planning capabilities. DriveVLM integrates a unique combination of reasoning modules for scene description, scene analysis, and hierarchical planning. Furthermore, recognizing the limitations of VLMs in spatial reasoning and heavy computational requirements, we propose DriveVLM-Dual, a hybrid system that synergizes the strengths of DriveVLM with the traditional autonomous driving pipeline. Experiments on both the nuScenes dataset and our SUP-AD dataset demonstrate the efficacy of DriveVLM and DriveVLM-Dual in handling complex and unpredictable driving conditions. Finally, we deploy the DriveVLM-Dual on a production vehicle, verifying it is effective in real-world autonomous driving environments.",Project Page: https://tsinghua-mars-lab.github.io/DriveVLM/,NaN,NaN,cs.CV,[cs.CV],"[https://arxiv.org/abs/2402.12289v5, https://arxiv.org/pdf/2402.12289v5]",ARXIV:2402.12289,540.0,41.0,78.0
378,http://arxiv.org/abs/2310.14414v2,https://arxiv.org/pdf/2310.14414v2,2024-06-20 19:36:38+00:00,2023-10-22 21:06:10+00:00,Vision Language Models in Autonomous Driving: A Survey and Outlook,"[Xingcheng Zhou, Mingyu Liu, Ekim Yurtsever, Bare Luka Zagar, Walter Zimmer, Hu Cao, Alois C. Knoll]","The

In [16]:
NS = {'opensearch': 'http://a9.com/-/spec/opensearch/1.1/'}
for y in range(2015, 2027):
    q = f'abs:"autonomous driving" AND submittedDate:[{y}01010000 TO {y}12312359]'
    q = f'abs:"autonomous driving" AND (abs:"E2E" OR abs:"End to End" OR "End-to-End")AND submittedDate:[{y}01010000 TO {y}12312359]'

    print(y, arxiv_total(q))
    time.sleep(3)

2015 0
2016 5
2017 10
2018 23
2019 35
2020 46
2021 68
2022 78
2023 105
2024 171
2025 324
2026 227


In [11]:
print(df['published'].dt.year.value_counts().sort_index())

published
2023     11
2024     51
2025    200
2026    120
Name: count, dtype: int64
